# Part 4 - Drift and Observability Using One UWB Beacon

This notebook demonstrates relative pose estimation with a single UWB anchor, fusing noisy odometry (VIO) with UWB range measurements using an SE(2) Extended Kalman Filter (EKF). We explore drift propagation and observability analysis in this configuration.

In [ ]:
# Cell 2: Imports

import numpy as np
import matplotlib.pyplot as plt

from manifpy import SE2

import sys
sys.path.insert(0, '../src')

from config import TrajectoryConfig, VIOConfig, UWBNoiseConfig, EKFConfig
from trajectory import generate_trajectory, sgal3_to_se2_poses
from vio_sensor import simulate_vio_odometry
from uwb import simulate_uwb_ranges, plot_anchor_positions, simulate_uwb_bearings
from plotting import plot_trajectory_2d, plot_error_ellipse
from ekf import simulate_ekf, simulate_ekf_with_bearing


In [ ]:
# Cell 3: Trajectory generation

# Parameters for trajectory generation of robot 1
traj_config = TrajectoryConfig(
    duration=100.0,
    timestep=0.05,   # 20 Hz odometry rate
    seed=42          # Reproducibility
)

# Generate trajectory
trajectory = generate_trajectory(
    duration=traj_config.duration,
    timestep=traj_config.timestep,
    seed=traj_config.seed
)

# Convert to SE(2) poses for visualization
poses_gt = sgal3_to_se2_poses(trajectory.X_RRi)

# Visualize ground truth trajectory
fig, ax = plt.subplots(figsize=(10, 8))
plot_trajectory_2d(ax, poses_gt, label='Ground Truth', color='k')
ax.set_title('Robot Trajectory (Perfect)')
plt.show()

In [ ]:
# Cell 4a: Exercise 1 - Robot to Anchor Relative Pose Estimation

# oc-todo: use the _2d_upf to implement a UPF that tracks the robot's pose relative to a single UWB anchor using range measurements and VIO odometry. A
# The goal is to demonstrate that this is unobserbale and you shoudl end up with multiple possible states.
# take 4, headings, 4 azimuth as initial configuration (16 particles.)

In [ ]:
# Cell 4b: Exercise 2 - Anchor to Robot Relative Pose Estimation with UPF

# Configuration (same as Exercise 1)
anchor_pos = np.array([5.0, 5.0])

vio_cfg = VIOConfig(sig_v=0.1, sig_w=0.05)
uwb_cfg = UWBNoiseConfig(sigma_range=0.1)

# Simulate noisy odometry and UWB ranges
poses_noisy, _ = simulate_vio_odometry(trajectory, vio_cfg)
ranges = simulate_uwb_ranges(poses_noisy, np.array([anchor_pos]), uwb_cfg)

# Run UPF to estimate beacon relative to robot trajectory
from ukf_helpers import estimate_beacon_with_upf
beacon_states, covariances = estimate_beacon_with_upf(poses_noisy, ranges, anchor_pos, vio_cfg, uwb_cfg)

# Convert beacon states from polar to Cartesian coordinates in robot frame
beacon_ests_cartesian = np.zeros_like(beacon_states)
for i in range(len(beacon_states)):
    r, theta, h = beacon_states[i]
    beacon_ests_cartesian[i, 0] = r * np.cos(theta)
    beacon_ests_cartesian[i, 1] = r * np.sin(theta)
    beacon_ests_cartesian[i, 2] = h

# Compute absolute beacon positions by transforming from robot frame to world frame
beacon_abs_positions = np.zeros((len(poses_noisy), 3))
for i in range(len(poses_noisy)):
    x_r, y_r, theta_r = poses_noisy[i]
    x_b_rel, y_b_rel, h_b = beacon_ests_cartesian[i]
    
    R = np.array([[np.cos(theta_r), -np.sin(theta_r)],
                  [np.sin(theta_r),  np.cos(theta_r)]])
    
    pos_abs = np.array([x_r, y_r]) + R @ np.array([x_b_rel, y_b_rel])
    beacon_abs_positions[i, :2] = pos_abs
    beacon_abs_positions[i, 2] = theta_r + h_b

# Plot results
fig, ax = plt.subplots(figsize=(10, 8))
plot_trajectory_2d(ax, poses_gt, label='Robot Trajectory', color='k')
ax.scatter([anchor_pos[0]], [anchor_pos[1]], c='red', marker='^', s=150, zorder=5, label='True Beacon')

plot_every = 10
for i in range(0, len(beacon_abs_positions), plot_every):
    x, y, theta = beacon_abs_positions[i]
    cov = covariances[i]
    
    r, theta_rel, h = beacon_states[i]
    P_beacon = np.array([[cov[0,0], cov[0,1]], [cov[1,0], cov[1,1]]])
    
    R_rot = np.array([[np.cos(theta), -np.sin(theta)],
                      [np.sin(theta),  np.cos(theta)]])
    P_world = R_rot @ P_beacon @ R_rot.T
    
    from matplotlib.patches import Ellipse
    ellipse = Ellipse((x, y), width=2*np.sqrt(P_world[0,0])*2.0,
                     height=2*np.sqrt(P_world[1,1])*2.0,
                     angle=np.degrees(np.arctan2(2*P_world[0,1], P_world[0,0]-P_world[1,1])/2)),
                     edgecolor='green', facecolor='none', linewidth=1.5)
    ax.add_patch(ellipse)

ax.scatter(beacon_abs_positions[-1, 0], beacon_abs_positions[-1, 1], 
           c='green', marker='o', s=80, zorder=6, label='UPF Estimate (final)')

ax.set_title('Exercise 2: Beacon Position Estimation with UPF')
ax.legend()
ax.axis('equal')
plt.show()

print(f"Final beacon estimate (absolute): [{beacon_abs_positions[-1, 0]:.3f}, {beacon_abs_positions[-1, 1]:.3f}]")
print(f"True beacon position: [{anchor_pos[0]}, {anchor_pos[1]}]")
print(f"Final error: {np.linalg.norm(beacon_abs_positions[-1, :2] - anchor_pos):.3f} m")

# Cell 5: Observability Analysis (Placeholder)

In [ ]:
# Cell 6: Exercise 3 - Range and Bearing Fusion

# Configuration (same as before but with bearing noise)
anchor_pos = np.array([5.0, 5.0])
vio_cfg = VIOConfig(sig_v=0.1, sig_w=0.05)
uwb_cfg = UWBNoiseConfig(sigma_range=0.1, sigma_bearing=0.02)  # Added bearing noise
ekf_cfg = EKFConfig()

# Simulate both range and bearing measurements
poses_noisy, odometry_traj = simulate_vio_odometry(trajectory, vio_cfg)

# Run EKF with range and bearing fusion
state_seq_rb, cov_seq_rb, t_seq_rb = simulate_ekf_with_bearing(
    trajectory,
    anchor_pos,
    ekf_cfg,
    (vio_cfg, uwb_cfg),
    seed=42
)

# Convert to arrays for plotting
poses_est_rb = np.array(state_seq_rb)

# Plot results and compare with range-only
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Range+bearing estimate
plot_trajectory_2d(ax1, poses_gt, label='Ground Truth', color='k')
plot_trajectory_2d(ax1, poses_est_rb, label='EKF (Range+Bearing)', color='g', linestyle='--')
plot_anchor_positions(ax1, np.array([anchor_pos]))

for i in range(0, len(poses_est_rb), plot_every):
    plot_error_ellipse(ax1, poses_est_rb[i], cov_seq_rb[i], color='green', n_std=2.0)

ax1.set_title('Range+Bearing EKF')
ax1.legend()

# Right: Compare uncertainty
fig2, ax3 = plt.subplots(figsize=(8, 6))
plot_trajectory_2d(ax3, poses_gt, label='Ground Truth', color='k')
ax3.set_title('Comparison of EKF Estimates')
ax3.legend()

plt.tight_layout()
plt.show()